In [17]:
import requests
import pandas as pd
import getpass  # Para ocultar a senha ao digitar no Jupyter

# Configuração da API
BASE_URL = ""  # Substitua pela URL da sua API


ModuleNotFoundError: No module named 'requests'

In [15]:
def autenticar():
    usuario = input("Usuário: ")
    senha = getpass.getpass("Senha: ")  # Oculta a senha ao digitar

    resposta = requests.post(f"{BASE_URL}/login", json={"username": usuario, "password": senha})
    
    if resposta.status_code == 200:
        token = resposta.json().get("access_token")
        print("Autenticação bem-sucedida!")
        return token
    else:
        print("Erro na autenticação:", resposta.text)
        return None


In [16]:
def obter_dados(token):
    """
    Obtém os dados da API utilizando o token de autenticação.
    Retorna um DataFrame com os dados obtidos.
    """
    headers = {"Authorization": f"Bearer {token}"}
    resposta = requests.get(f"{BASE_URL}/dados", headers=headers)

    if resposta.status_code == 200:
        dados = resposta.json()
        return pd.DataFrame(dados)  # Converte a resposta JSON para DataFrame
    else:
        print("Erro ao obter os dados:", resposta.text)
        return None


In [ ]:
def limpar_dados(df):
    """
    Converte colunas para tipos numéricos e trata valores nulos.
    """
    if df is not None:
        # Convertendo colunas numéricas
        df["a1"] = pd.to_numeric(df["a1"], errors="coerce")
        df["a2"] = pd.to_numeric(df["a2"], errors="coerce")
        df["a3"] = pd.to_numeric(df["a3"], errors="coerce")

        # Convertendo data/hora
        df["data_hora"] = pd.to_datetime(df["data_hora"], errors="coerce")

        # Removendo valores nulos
        df.dropna(inplace=True)

        print("Dados limpos com sucesso!")
        return df
    else:
        print("Nenhum dado para limpar.")
        return None


In [ ]:
def postar_dados(df, token):
    """
    Envia os dados limpos de volta para a API.
    """
    if df is not None and not df.empty:
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        dados_json = df.to_dict(orient="records")  # Converte DataFrame para lista de dicionários

        resposta = requests.post(f"{BASE_URL}/enviar_dados", json=dados_json, headers=headers)

        if resposta.status_code == 201:
            print("Dados enviados com sucesso!")
        else:
            print("Erro ao enviar os dados:", resposta.text)
    else:
        print("Nenhum dado válido para enviar.")


In [ ]:
# Limpeza dos dados
if df is not None:
    df_limpo = limpar_dados(df)

    # Envio dos dados limpos para a API
    postar_dados(df_limpo, token)
